# 📊 End-to-End Benchmarking, Results Ledger & Dossier Analytics

This tutorial demonstrates how to leverage the **NeuralOps Results Ledger** and benchmarking harness to execute benchmarks, track performance telemetry across versions, query historical records via SQLite, and generate automated module dossiers.

### Core Topics Covered:
1. **Executing Empirical Benchmarks**: Run multi-mode throughput and latency benchmarks comparing linear attention vs. naive PyTorch quadratic baselines.
2. **Persistent Telemetry Logging (`ResultsLedger`)**: Record benchmarks with comprehensive hardware metadata (SM version, arithmetic intensity, memory footprint).
3. **Interactive SQLite Ledger Querying**: Run complex SQL queries to filter Pareto-optimal configurations.
4. **Automated Architectural Dossiers**: Generate Markdown dossiers and summary cards for publishing research artifacts.
5. **Multi-Version Performance Tracking**: Compare speedups across library releases (e.g. `v1.3.6` vs `v1.3.7`).


In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import torch

import neuralops
from neuralops import (
    get_default_ledger,
    ATTENTION_REGISTRY,
    COMPRESSOR_REGISTRY,
    profile_roofline,
)

print(f"NeuralOps Version: {neuralops.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 1. Inspecting the Persistent Ledger Index

The `ResultsLedger` maintains dual-layer storage:
- **Append-only JSONL logs**: Guaranteeing non-destructive append operations.
- **SQLite Index (`index.db`)**: Providing high-speed indexing across module names, categories, git commits, and versions.


In [ ]:
ledger = get_default_ledger()

print(f"Ledger Root Directory: {ledger.root_dir}")
print(f"SQLite Path:           {ledger.sqlite_path}")
print(f"JSONL Path:            {ledger.jsonl_path}")

stats = ledger.get_summary_stats()
print("\nLedger Global Telemetry Summary:")
print(f"  Total Historical Records: {stats['total_records']}")
print(f"  Active Records:           {stats['active_records']}")
print(f"  Distinct Modules:         {stats['distinct_modules']}")
print(f"  Categories Tracked:       {stats['by_category']}")
print(f"  Versions Tracked:         {stats['by_version']}")


## 2. Querying the SQLite Index Directly

We can leverage the high-level `ledger.query()` API or raw SQL to analyze performance across canonical attention mechanisms.


In [ ]:
# Query top attention mechanisms by speedup vs PyTorch
top_attns = ledger.query(category="attention", limit=10)

records_data = []
for r in top_attns:
    records_data.append({
        "Module": r.module_name,
        "Version": r.version,
        "Latency (ms)": round(r.latency_ms, 3) if r.latency_ms else None,
        "Speedup (x)": round(r.speedup_vs_pytorch, 2) if r.speedup_vs_pytorch else None,
        "Peak Memory (MB)": round(r.peak_memory_mb, 1) if r.peak_memory_mb else None,
        "Commit": r.git_commit[:8] if r.git_commit else "N/A",
    })

df = pd.DataFrame(records_data)
print(df.to_string(index=False))


## 3. Visualizing Speedup Distributions Across Attention Families

Let us visualize the empirical speedup distribution across benchmarked linear attention modules.


In [ ]:
# Extract speedups for plotting
speedup_records = ledger.query(category="attention", limit=20)
names = [r.module_name for r in speedup_records if r.speedup_vs_pytorch and r.speedup_vs_pytorch > 1.0][:8]
speedups = [r.speedup_vs_pytorch for r in speedup_records if r.speedup_vs_pytorch and r.speedup_vs_pytorch > 1.0][:8]

if names:
    plt.figure(figsize=(10, 4.5), dpi=100)
    bars = plt.barh(names, speedups, color='royalblue', edgecolor='navy', height=0.6)
    plt.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='PyTorch Reference Baseline (1.0x)')
    plt.xlabel('Measured Speedup (x)', fontsize=12)
    plt.title('Empirical Speedup Across Canonical Attention Modules', fontsize=13, fontweight='bold')
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.legend()
    for bar in bars:
        w = bar.get_width()
        plt.text(w + 0.5, bar.get_y() + bar.get_height()/2, f'{w:.1f}x', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient records for bar plot.")


## 4. Module Dossier Retrieval & Analysis

Each module indexed in the ledger can produce a comprehensive research dossier containing run histories, parameter settings, and hardware execution metrics.


In [ ]:
# Inspect dossier for a flagship module: gated_delta_net
dossier = ledger.get_module_dossier("gated_delta_net")
if dossier:
    print(f"=== Module Dossier: {dossier.module_name} ===")
    print(f"Category: {dossier.category}")
    print(f"Total Historical Runs: {len(dossier.historical_runs)}")
    
    latest_run = dossier.latest_record
    if latest_run:
        print(f"\nLatest Run Details ({latest_run.run_id}):")
        print(f"  Version:               {latest_run.version}")
        print(f"  Measured Speedup:      {latest_run.speedup_vs_pytorch:.2f}x")
        print(f"  Measured Latency:      {latest_run.latency_ms:.3f} ms")
        print(f"  Peak Memory Footprint: {latest_run.peak_memory_mb:.1f} MB")
        print(f"  Hardware Device:       {latest_run.device}")
else:
    print("Dossier for 'gated_delta_net' not found.")


## 5. Direct SQLite Analytical Queries

Because the index is stored in a relational SQLite table, advanced SQL aggregations and queries can be executed directly for telemetry reporting.


In [ ]:
conn = sqlite3.connect(ledger.sqlite_path)
cursor = conn.cursor()

# Query average speedup and latency grouped by category
query = """
SELECT 
    category,
    COUNT(*) as record_count,
    ROUND(AVG(speedup_vs_pytorch), 2) as avg_speedup,
    ROUND(MAX(speedup_vs_pytorch), 2) as max_speedup,
    ROUND(AVG(latency_ms), 3) as avg_latency_ms
FROM benchmark_results
WHERE active = 1 AND speedup_vs_pytorch IS NOT NULL
GROUP BY category
ORDER BY record_count DESC;
"""

sql_df = pd.read_sql_query(query, conn)
conn.close()

print("Category Performance Aggregates from SQLite:")
print(sql_df.to_string(index=False))
print("\nEnd-to-End Benchmarking & Results Ledger walkthrough completed successfully [100% OK]!")
